# Interval Rep Charts and Values

This notebook visualizes interval-rep quality for:
1. Rep power consistency
2. HR recovery between reps
3. Cardiac drift across reps
4. Peak HR progression
5. Rep pace consistency

You can analyze one specific run or all interval runs together.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')

cwd = Path.cwd()
candidate_report_dirs = [
    cwd / 'reports' / 'interval',
    cwd.parent / 'reports' / 'interval',
]

report_dir = next((p for p in candidate_report_dirs if (p / 'interval_level_dataset.csv').exists()), None)
if report_dir is None:
    raise FileNotFoundError(f'Could not find reports/interval from {cwd}')

interval_level_path = report_dir / 'interval_level_dataset.csv'
workout_level_path = report_dir / 'interval_workouts_dataset.csv'

interval_df = pd.read_csv(interval_level_path)
workout_df = pd.read_csv(workout_level_path) if workout_level_path.exists() else pd.DataFrame()

if 'date' in interval_df.columns:
    interval_df['date'] = pd.to_datetime(interval_df['date'], errors='coerce')
if not workout_df.empty and 'date' in workout_df.columns:
    workout_df['date'] = pd.to_datetime(workout_df['date'], errors='coerce')

interval_df = interval_df[interval_df['session_type'].str.lower().eq('interval')].copy()
if interval_df.empty:
    raise ValueError('No interval rows found in interval_level_dataset.csv')

print(f'Using report folder: {report_dir}')
print(f'Interval runs available: {interval_df["file"].nunique()}')

In [ ]:
# Set to a specific filename to analyze one run, or keep None for all interval runs
SELECTED_RUN = None
# Example: SELECTED_RUN = '2026-05-26_tuesday_interval.fit'

available_runs = sorted(interval_df['file'].dropna().unique())
print('Available interval runs:')
for run_name in available_runs:
    print(' -', run_name)

if SELECTED_RUN is None:
    plot_df = interval_df.copy()
    print('\nMode: ALL interval runs')
else:
    plot_df = interval_df[interval_df['file'].eq(SELECTED_RUN)].copy()
    if plot_df.empty:
        raise ValueError(f'SELECTED_RUN not found: {SELECTED_RUN}')
    print(f'\nMode: Single run -> {SELECTED_RUN}')

plot_df = plot_df.sort_values(['file', 'interval_index'])
plot_df[['avg_power', 'hr_recovery_60s_bpm', 'avg_hr', 'max_hr', 'pace_min_per_km']] = plot_df[['avg_power', 'hr_recovery_60s_bpm', 'avg_hr', 'max_hr', 'pace_min_per_km']].apply(pd.to_numeric, errors='coerce')

display(plot_df.head(10))

In [ ]:
def cv_percent(series: pd.Series) -> float:
    x = pd.to_numeric(series, errors='coerce').dropna()
    if len(x) < 2 or float(x.mean()) == 0.0:
        return np.nan
    return float((x.std(ddof=1) / x.mean()) * 100.0)

def rep_drift(series: pd.Series) -> float:
    x = pd.to_numeric(series, errors='coerce').dropna()
    if len(x) < 2:
        return np.nan
    return float(x.iloc[-1] - x.iloc[0])

summary = (
    plot_df.groupby('file', as_index=False)
    .apply(
        lambda g: pd.Series({
            'rep_count': int(g['interval_index'].nunique()),
            'rep_power_consistency_cv_pct': cv_percent(g['avg_power']),
            'hr_recovery_between_reps_bpm_mean': float(pd.to_numeric(g['hr_recovery_60s_bpm'], errors='coerce').mean()),
            'cardiac_drift_across_reps_bpm': rep_drift(g['avg_hr']),
            'peak_hr_progression_bpm': rep_drift(g['max_hr']),
            'rep_pace_consistency_cv_pct': cv_percent(g['pace_min_per_km']),
        })
    )
    .reset_index(drop=True)
)

summary = summary.sort_values('file').reset_index(drop=True)
display(summary)

In [ ]:
metrics = [
    ('avg_power', 'Rep Power (W)', 'Rep power consistency'),
    ('hr_recovery_60s_bpm', 'HR Recovery 60s (bpm)', 'HR recovery between reps'),
    ('avg_hr', 'Avg HR (bpm)', 'Cardiac drift across reps'),
    ('max_hr', 'Peak HR (bpm)', 'Peak HR progression'),
    ('pace_min_per_km', 'Pace (min/km)', 'Rep pace consistency'),
]

fig, axes = plt.subplots(3, 2, figsize=(16, 14))
axes = axes.flatten()

for i, (col, y_label, title) in enumerate(metrics):
    ax = axes[i]

    if SELECTED_RUN is not None:
        g = plot_df.sort_values('interval_index')
        y = pd.to_numeric(g[col], errors='coerce')

        if col == 'avg_hr':
            y = y - y.iloc[0] if len(y.dropna()) else y
            y_label = 'Drift From Rep 1 (bpm)'

        ax.plot(g['interval_index'], y, marker='o', linewidth=2)
    else:
        for run_name, g in plot_df.groupby('file'):
            g = g.sort_values('interval_index')
            y = pd.to_numeric(g[col], errors='coerce')

            if col == 'avg_hr' and len(y.dropna()) > 0:
                y = y - y.iloc[0]
                y_label = 'Drift From Rep 1 (bpm)'

            ax.plot(g['interval_index'], y, marker='o', linewidth=1, alpha=0.35, color='gray')

        rep_agg = plot_df.groupby('interval_index')[col].agg(['mean', 'std']).reset_index()
        mean_y = pd.to_numeric(rep_agg['mean'], errors='coerce')

        if col == 'avg_hr' and len(mean_y.dropna()) > 0:
            mean_y = mean_y - mean_y.iloc[0]
            y_label = 'Drift From Rep 1 (bpm)'

        ax.plot(rep_agg['interval_index'], mean_y, marker='o', linewidth=2.5, color='tab:red', label='Mean')

        std_y = pd.to_numeric(rep_agg['std'], errors='coerce').fillna(0.0)
        ax.fill_between(
            rep_agg['interval_index'],
            mean_y - std_y,
            mean_y + std_y,
            color='tab:red',
            alpha=0.15,
            label='±1 SD'
        )
        ax.legend(loc='best')

    ax.set_title(title)
    ax.set_xlabel('Rep #')
    ax.set_ylabel(y_label)

axes[-1].axis('off')
mode_text = SELECTED_RUN if SELECTED_RUN is not None else 'All interval runs'
fig.suptitle(f'Interval Rep Metrics - {mode_text}', fontsize=16, y=1.02)
fig.tight_layout()
plt.show()